# Entrega 3: Modelo predictivo lineal 
### Detección de Tráfico Malicioso en Redes IoT
**Santiago Vieira Ceballos — Sara Franco Taborda — Sara Medina Molina**

## 1. Preparación de los datos

En esta sección se carga el dataset y se replica la limpieza realizada en la
Entrega 2: eliminación de duplicados, corrección de tipos de datos y descarte
de variables irrelevantes (`src_ip`, `dst_ip` por ser identificadores sin
valor predictivo, y `type` por ser redundante con la variable objetivo `label`).

Posteriormente se separan las características (X) de la variable objetivo (y),
y se realiza la partición en conjuntos de entrenamiento (80%) y prueba (20%)
con `stratify=y` para preservar la proporción de clases dado el desbalance
identificado en la entrega anterior.

Las transformaciones específicas (imputación, escalamiento, codificación) se
implementarán dentro del pipeline en el punto 3 para evitar fuga de datos
durante la validación cruzada.

In [19]:
from pathlib import Path
import pandas as pd
import numpy as np

DATA_DIR = Path().resolve().parent / "Data"
data_file = "train_test_network.csv"
data_path = DATA_DIR / data_file

# Carga del dataset
df = pd.read_csv(data_path, na_values=[' '])
print(f'Dimensiones iniciales: {df.shape}')

# Corrección de tipos: variables que son códigos categóricos pero están como int
df = df.astype({
    'src_port'   : 'object',
    'dst_port'   : 'object',
    'dns_qclass' : 'object',
    'dns_qtype'  : 'object',
    'dns_rcode'  : 'object',
    'label'      : 'object'
})

# Eliminación de duplicados
df = df.drop_duplicates()
print(f'Dimensiones tras eliminar duplicados: {df.shape}')

# Descarte de variables irrelevantes:
# - src_ip y dst_ip son identificadores de red sin valor predictivo
# - type es redundante con label (ambas codifican lo mismo, una binaria y otra multiclase)
df = df.drop(columns=['src_ip', 'dst_ip', 'type'])
print(f'Dimensiones finales: {df.shape}')

df.head()

Dimensiones iniciales: (211043, 44)
Dimensiones tras eliminar duplicados: (190474, 44)
Dimensiones finales: (190474, 41)


,src_port,dst_port,proto,service,duration,src_bytes,dst_bytes,conn_state,missed_bytes,src_pkts,...,http_request_body_len,http_response_body_len,http_status_code,http_user_agent,http_orig_mime_types,http_resp_mime_types,weird_name,weird_addl,weird_notice,label
0,4444,49178,tcp,-,290.371539,101568,2592,OTH,0,108,...,0,0,0,-,-,-,-,-,-,1
1,49180,8080,tcp,-,0.000102,0,0,REJ,0,1,...,0,0,0,-,-,-,-,-,-,1
2,49180,8080,tcp,-,0.000148,0,0,REJ,0,1,...,0,0,0,-,-,-,-,-,-,1
3,49180,8080,tcp,-,0.000113,0,0,REJ,0,1,...,0,0,0,-,-,-,-,-,-,1
4,49180,8080,tcp,-,0.000130,0,0,REJ,0,1,...,0,0,0,-,-,-,-,-,-,1


### Separación de características y variable objetivo

La variable a predecir es `label`, una variable binaria que indica si un flujo
de red es benigno (0) o malicioso (1). Por tratarse de un problema de
clasificación binaria, se utilizará un modelo de **Regresión Logística
regularizada** en el punto 2.

In [20]:
# Separación de características (X) y variable objetivo (y)
X = df.drop(columns=['label'])
y = df['label'].astype(int)  # se convierte a int para el modelo

print(f'Dimensiones de X: {X.shape}')
print(f'Dimensiones de y: {y.shape}')

# Verificación del balance de clases
print('\nDistribución de la variable objetivo:')
print(y.value_counts())
print(f'\nProporción clase minoritaria: {y.value_counts(normalize=True).min():.3f}')

Dimensiones de X: (190474, 40)
Dimensiones de y: (190474,)

Distribución de la variable objetivo:
label
1    148434
0     42040
Name: count, dtype: int64

Proporción clase minoritaria: 0.221


### Partición en entrenamiento y prueba

Se realiza una partición 80/20 con `random_state=42` para garantizar
reproducibilidad. Se utiliza `stratify=y` para que ambos subconjuntos
mantengan la misma proporción de clases que el dataset original, lo cual
es importante dado el desbalance observado.

In [21]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f'Tamaño del conjunto de entrenamiento: {X_train.shape}')
print(f'Tamaño del conjunto de prueba: {X_test.shape}')

# Verificación de la estratificación
print('\nProporción de clases en entrenamiento:')
print(y_train.value_counts(normalize=True))
print('\nProporción de clases en prueba:')
print(y_test.value_counts(normalize=True))

Tamaño del conjunto de entrenamiento: (152379, 40)
Tamaño del conjunto de prueba: (38095, 40)

Proporción de clases en entrenamiento:
label
1    0.779287
0    0.220713
Name: proportion, dtype: float64

Proporción de clases en prueba:
label
1    0.779289
0    0.220711
Name: proportion, dtype: float64


### Identificación de variables numéricas y categóricas

Se identifican los dos tipos de variables presentes en `X_train` para
aplicarles transformaciones independientes en el pipeline del punto 3.
La identificación se hace sobre `X_train` (no sobre `X` completo) para
mantener la separación estricta entre los datos de entrenamiento y prueba.

In [22]:
# Reclasificación de puertos como numéricos para evitar explosión dimensional
# Aunque src_port y dst_port son códigos, su alta cardinalidad (miles de
# valores únicos debido a puertos efímeros) hace inviable la codificación
# OneHot. Los tratamos como numéricos aprovechando que existe una relación
# ordinal parcial entre rangos de puertos (well-known, registered, ephemeral).
X_train = X_train.astype({'src_port': 'int64', 'dst_port': 'int64'})
X_test = X_test.astype({'src_port': 'int64', 'dst_port': 'int64'})

# Identificación de variables numéricas y categóricas
numeric_features = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X_train.select_dtypes(include=['object']).columns.tolist()

print(f'Variables numéricas ({len(numeric_features)}):')
print(numeric_features)
print(f'\nVariables categóricas ({len(categorical_features)}):')
print(categorical_features)

Variables numéricas (13):
['src_port', 'dst_port', 'duration', 'src_bytes', 'dst_bytes', 'missed_bytes', 'src_pkts', 'src_ip_bytes', 'dst_pkts', 'dst_ip_bytes', 'http_request_body_len', 'http_response_body_len', 'http_status_code']

Variables categóricas (27):
['proto', 'service', 'conn_state', 'dns_query', 'dns_qclass', 'dns_qtype', 'dns_rcode', 'dns_AA', 'dns_RD', 'dns_RA', 'dns_rejected', 'ssl_version', 'ssl_cipher', 'ssl_resumed', 'ssl_established', 'ssl_subject', 'ssl_issuer', 'http_trans_depth', 'http_method', 'http_uri', 'http_version', 'http_user_agent', 'http_orig_mime_types', 'http_resp_mime_types', 'weird_name', 'weird_addl', 'weird_notice']


## 2. Implementación de un modelo lineal regularizado

La variable a predecir es `label`, una variable **binaria** que indica si un
flujo de red corresponde a tráfico normal (0) o malicioso (1). Por tratarse
de un problema de **clasificación binaria**, el modelo lineal regularizado
adecuado es la **Regresión Logística regularizada** (`LogisticRegression`
de scikit-learn).

A diferencia de los modelos `Ridge` y `Lasso` —que se usan para problemas de
regresión con variable objetivo continua— `LogisticRegression` modela la
probabilidad de pertenencia a una clase mediante la función logística, y
permite aplicar regularización L1, L2 o Elastic Net para controlar la
complejidad del modelo y reducir la varianza.

### Hiperparámetros relevantes

`LogisticRegression` tiene tres hiperparámetros principales para sintonizar:

- **`penalty`**: tipo de regularización aplicada. Puede ser `'l1'`, `'l2'` o
  `'elasticnet'`. La penalización `'l2'` reduce el tamaño de todos los
  coeficientes, mientras que `'l1'` puede llevar coeficientes exactamente a
  cero (selección implícita de características). `'elasticnet'` combina ambas.

- **`C`**: es el inverso del factor de regularización ($C = 1/\lambda$). Puede
  variar en el rango $(0, \infty)$. Mientras **más grande** sea `C`, **menor**
  es la penalización sobre los coeficientes (modelo más complejo); mientras
  **más pequeño**, **mayor** la regularización (modelo más simple).

- **`l1_ratio`**: solo aplica si `penalty='elasticnet'`. Define el peso
  relativo entre L1 y L2.

Para esta entrega se utilizará `penalty='elasticnet'` con el solver `'saga'`,
que soporta los tres tipos de regularización y permite explorar el espacio
completo de hiperparámetros durante la sintonización en el punto 4. Además,
se usará `class_weight='balanced'` para abordar el desbalance de clases
identificado anteriormente.

In [23]:
from sklearn.linear_model import LogisticRegression

# Instanciación del modelo lineal regularizado
# - penalty='elasticnet': permite combinar regularización L1 y L2
# - solver='saga': único solver compatible con elasticnet y datasets grandes
# - class_weight='balanced': ajusta automáticamente los pesos inversamente
#   proporcionales a la frecuencia de cada clase, para manejar el desbalance
# - max_iter=2000: se incrementa para garantizar convergencia
# - random_state=42: para reproducibilidad
# Los hiperparámetros C y l1_ratio se sintonizarán en el punto 4

modelo = LogisticRegression(
    penalty='elasticnet',
    solver='saga',
    l1_ratio=0.5,        # valor inicial, se sintonizará después
    C=1.0,               # valor inicial, se sintonizará después
    class_weight='balanced',
    max_iter=2000,
    random_state=42
)

modelo

,penalty,'elasticnet'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,'balanced'
,random_state,42
,solver,'saga'
,max_iter,2000
,multi_class,'deprecated'


## 3. Pipeline integrado de preprocesamiento y modelado

En esta sección se construye un pipeline que integra todos los pasos de
preprocesamiento junto con el modelo definido en el punto 2. El uso de
`Pipeline` y `ColumnTransformer` evita la **fuga de datos** durante la
validación cruzada y la sintonización de hiperparámetros, garantizando
que las transformaciones se ajusten únicamente con los datos de
entrenamiento de cada fold.

El pipeline incluye los siguientes procesos:

1. **Imputación**: aunque el dataset no tiene valores nulos, se incluye
   como medida de robustez ante datos futuros.
2. **Estandarización** de variables cuantitativas con `StandardScaler`,
   necesaria para que la regularización penalice de forma equitativa
   los coeficientes de variables con escalas distintas.
3. **Codificación** de variables cualitativas con `OneHotEncoder`, que
   transforma cada categoría en una columna binaria.
4. **Selección de características** con `SelectKBest` usando el estadístico
   ANOVA F, para reducir la dimensionalidad y quedarse con las variables
   más relevantes.
5. **Balanceo de clases** mediante el parámetro `class_weight='balanced'`
   ya configurado en el modelo del punto 2, que ajusta los pesos
   inversamente proporcionales a la frecuencia de cada clase.

In [24]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

# Transformador para variables numéricas:
# - Imputación con mediana (robusta a outliers, identificados en la Entrega 2)
# - Estandarización a media 0 y desviación 1
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Transformador para variables categóricas:
# - Imputación con la categoría más frecuente
# - Codificación OneHot, ignorando categorías nuevas que aparezcan en test
#   y usando sparse_output=False para compatibilidad con SelectKBest
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Combinación de ambos transformadores en un único ColumnTransformer
preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

preprocessor

,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,missing_values,nan
,strategy,'median'
,fill_value,None


### Selección de características

Tras la codificación OneHot, el número de columnas crece considerablemente
(cada categoría se convierte en una columna binaria). Para evitar problemas
de dimensionalidad y reducir el ruido, se aplica `SelectKBest` con la
prueba estadística **ANOVA F** (`f_classif`), que mide la dependencia
lineal entre cada característica y la variable objetivo.

El número óptimo de características (`k`) se sintonizará en el punto 4
junto con los demás hiperparámetros del modelo.

In [25]:
from sklearn.feature_selection import SelectKBest, f_classif

# Pipeline completo: preprocesamiento + selección de características + modelo
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('feature_selection', SelectKBest(score_func=f_classif, k=20)),  # k se sintonizará
    ('classifier', modelo)  # modelo definido en el punto 2
])

pipeline

,steps,"[('preprocessor', ...), ('feature_selection', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


### Verificación del pipeline

Antes de proceder con la sintonización de hiperparámetros, se ajusta el
pipeline con los hiperparámetros por defecto para verificar que todos
los pasos funcionan correctamente y no hay errores en las dimensiones
o tipos de datos.

In [26]:
# Ajuste de prueba con los datos de entrenamiento
pipeline.fit(X_train, y_train)

# Verificación: número de características antes y después de la selección
n_features_before = pipeline.named_steps['preprocessor'].transform(X_train).shape[1]
n_features_after = pipeline.named_steps['feature_selection'].get_support().sum()

print(f'Características tras preprocesamiento (OneHot): {n_features_before}')
print(f'Características tras selección (SelectKBest): {n_features_after}')
print(f'\nPipeline ajustado correctamente.')

Características tras preprocesamiento (OneHot): 944
Características tras selección (SelectKBest): 20

Pipeline ajustado correctamente.


## 4. Sintonización de hiperparámetros con validación cruzada

La rúbrica de la entrega exige utilizar una técnica de sintonización de
hiperparámetros **distinta de `GridSearchCV`**. Por ello se emplea
`RandomizedSearchCV`, que en lugar de probar exhaustivamente todas las
combinaciones de la grilla, **muestrea aleatoriamente un número fijo de
combinaciones** del espacio de búsqueda.

### Ventajas de `RandomizedSearchCV` sobre `GridSearchCV`

1. **Eficiencia computacional**: con un dataset grande como el nuestro
   (~190K registros) y un pipeline costoso, evaluar todas las combinaciones
   posibles es inviable. `RandomizedSearchCV` permite controlar el costo
   con el parámetro `n_iter`.

2. **Mejor cobertura del espacio continuo**: para hiperparámetros como `C`
   que viven en un rango continuo, muestrear aleatoriamente en escala
   logarítmica suele encontrar mejores valores que una grilla discreta
   fija (Bergstra & Bengio, 2012).

3. **Permite explorar más hiperparámetros simultáneamente** sin que el
   costo crezca exponencialmente.

### Hiperparámetros a sintonizar

- **`classifier__C`**: inverso del factor de regularización del modelo
  logístico. Se muestrea en escala logarítmica entre $10^{-3}$ y $10^{2}$.
- **`classifier__l1_ratio`**: peso relativo entre regularización L1 y L2
  en Elastic Net. Se muestrea uniformemente entre 0 y 1.
- **`feature_selection__k`**: número de características seleccionadas por
  `SelectKBest`. Se prueban valores entre 20 y 100.

### Métrica de evaluación

Como el problema es de clasificación binaria con clases desbalanceadas,
se utiliza **`f1_score`** como métrica de optimización en la validación
cruzada. F1 combina precisión y recall, lo que la hace robusta al
desbalance de clases y consistente con el objetivo del problema:
detectar correctamente el tráfico malicioso minimizando tanto falsos
positivos como falsos negativos.

In [31]:
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from scipy.stats import loguniform, uniform

# Espacio de búsqueda de hiperparámetros
param_distributions = {
    'classifier__C': loguniform(1e-3, 1e2),
    'classifier__l1_ratio': uniform(0, 1),
    'feature_selection__k': [20, 40, 60, 80, 100]
}

# Validación cruzada estratificada con 3 folds (en lugar de 5) para acelerar
# la búsqueda. Sigue siendo robusta dado el tamaño del dataset.
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

# RandomizedSearchCV con n_iter=5 (en lugar de 10) para reducir el tiempo
# total. Total: 5 combinaciones × 3 folds = 15 entrenamientos.
random_search = RandomizedSearchCV(
    estimator=pipeline,
    param_distributions=param_distributions,
    n_iter=5,
    cv=cv,
    scoring='f1',
    refit=True,
    n_jobs=1,
    random_state=42,
    verbose=2
)

random_search

,estimator,Pipeline(step...ver='saga'))])
,param_distributions,"{'classifier__C': <scipy.stats....0015723325E80>, 'classifier__l1_ratio': <scipy.stats....001574B0CCC20>, 'feature_selection__k': [20, 40, ...]}"
,n_iter,5
,scoring,'f1'
,n_jobs,1
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,2
,pre_dispatch,'2*n_jobs'
,random_state,42
,error_score,nan


### Justificación del número de iteraciones y folds

Se utilizan `n_iter=5` combinaciones de hiperparámetros y `cv=3` folds
debido al alto costo computacional del pipeline (cada entrenamiento del
modelo logístico con `solver='saga'` sobre ~152.000 muestras toma varios
minutos). Esta configuración representa un balance razonable entre
exploración del espacio de hiperparámetros y tiempo de cómputo viable
en hardware estándar. Aunque `RandomizedSearchCV` no garantiza encontrar
el óptimo global, el muestreo log-uniforme sobre `C` cubre 5 órdenes de
magnitud, lo que asegura una exploración razonable del rango de
regularización.

In [ ]:
random_search.fit(X_train, y_train)

print(f'\nMejor score F1 de validación cruzada: {random_search.best_score_:.4f}')
print(f'\nMejores hiperparámetros encontrados:')
for param, value in random_search.best_params_.items():
    if isinstance(value, float):
        print(f'  {param}: {value:.4f}')
    else:
        print(f'  {param}: {value}')

Fitting 3 folds for each of 5 candidates, totalling 15 fits


c:\Users\saram\repos\Proyecto_analisis\.venv\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


[CV] END classifier__C=0.0745934328572655, classifier__l1_ratio=0.9507143064099162, feature_selection__k=60; total time= 4.9min
